In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
from pathlib import Path

SWEEP = Path('/n/holylfs06/LABS/kempner_project_b/Lab/dendritic/adivm/dendritic-modeling/outputs/interpretability_results/selectivity_sweep')
FIG_DIR = Path('figures'); FIG_DIR.mkdir(exist_ok=True)

GRID    = '#e1e0d9'
INK2    = '#52514e'
MUTED   = '#898781'
SURFACE = '#fcfcfb'

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#c3c2b7', 'axes.linewidth': 0.8,
    'grid.color': GRID, 'grid.linewidth': 0.5, 'axes.grid': True, 'axes.axisbelow': True,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'xtick.labelsize': 9,  'ytick.labelsize': 9,
    'axes.labelcolor': INK2, 'axes.labelsize': 10,
    'legend.framealpha': 0.9, 'legend.edgecolor': GRID, 'legend.fontsize': 9,
    'font.family': 'sans-serif',
})

# Series definitions: (tag_prefix, label, color, linestyle)
SERIES = [
    ('fc7_2x2',   '2x2 (6 branches)',  '#2a78d6', '-',  'o'),
    ('fc7_3x3x3', '3x3x3 (39 branches)', '#1baf7a', '-',  's'),
    ('fc7_point', 'Point neuron',       '#eda100', '--', '^'),
]
NE_VALUES = [32, 64, 128, 256, 512, 1024, 2048, 4096]

In [ ]:
# Load all summary.json files
results = {}   # tag -> summary dict
missing = []
for prefix, label, color, ls, marker in SERIES:
    for ne in NE_VALUES:
        tag = f"{prefix}_ne{ne}"
        summary_path = SWEEP / tag / 'summary.json'
        if summary_path.exists():
            with open(summary_path) as f:
                results[tag] = json.load(f)
        else:
            missing.append(tag)

print(f"Loaded {len(results)} results")
if missing:
    print(f"Missing ({len(missing)}): {missing}")

## Figure 1 — Median selectivity index vs N_E by morphology
Does more neurons or more dendritic branches lead to greater class specialization?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric, ylabel in [
    (axes[0], 'si_median', 'Median selectivity index S'),
    (axes[1], 'si_mean',   'Mean selectivity index S'),
]:
    for prefix, label, color, ls, marker in SERIES:
        xs, ys = [], []
        for ne in NE_VALUES:
            tag = f"{prefix}_ne{ne}"
            if tag in results:
                xs.append(ne)
                ys.append(results[tag][metric])
        if xs:
            ax.plot(xs, ys, color=color, ls=ls, marker=marker,
                    ms=6, lw=1.8, label=label, zorder=3)

    ax.set_xscale('log', base=2)
    ax.set_xlabel('Number of excitatory neurons N_E')
    ax.set_ylabel(ylabel)
    ax.set_xticks(NE_VALUES)
    ax.set_xticklabels([str(n) for n in NE_VALUES])
    ax.legend(title='Morphology')

plt.suptitle('Class selectivity index vs. model size, by dendritic morphology',
             fontsize=11, color='#0b0b0b', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'sel_sweep_fig1_si_vs_ne.png', bbox_inches='tight', dpi=200)
plt.show()

## Figure 2 — Selectivity vs. validation accuracy
Does higher selectivity come at a cost to accuracy, or do they co-occur?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for prefix, label, color, ls, marker in SERIES:
    xs, ys = [], []
    for ne in NE_VALUES:
        tag = f"{prefix}_ne{ne}"
        if tag in results:
            xs.append(results[tag]['si_median'])
            ys.append(results[tag]['val_acc'] * 100)
    if xs:
        ax.scatter(xs, ys, color=color, marker=marker, s=60,
                   label=label, zorder=3, linewidths=0.5, edgecolors='white')
        # annotate N_E
        for ne, x, y in zip(NE_VALUES, xs, ys):
            ax.annotate(str(ne), (x, y), textcoords='offset points',
                        xytext=(4, 3), fontsize=6, color=INK2)

ax.set_xlabel('Median selectivity index S (lifetime sparseness; 0 = uniform, 1 = single class)')
ax.set_ylabel('Top-1 validation accuracy (%)')
ax.set_title('Selectivity vs. validation accuracy across all sweep models', color='#0b0b0b')
ax.legend(title='Morphology')
plt.tight_layout()
plt.savefig(FIG_DIR / 'sel_sweep_fig2_si_vs_acc.png', bbox_inches='tight', dpi=200)
plt.show()

## Figure 3 — Selectivity index distributions per morphology (violin)
Full distribution shapes across all N_E, split by morphology.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, (prefix, label, color, ls, marker) in zip(axes, SERIES):
    all_si, all_ne = [], []
    for ne in NE_VALUES:
        tag = f"{prefix}_ne{ne}"
        si_path = SWEEP / tag / 'selectivity_index.npy'
        if si_path.exists():
            si = np.load(si_path)
            all_si.append(si)
            all_ne.append(ne)

    if not all_si:
        ax.set_title(f'{label} (no data)', color='#0b0b0b')
        continue

    parts = ax.violinplot(all_si, positions=range(len(all_ne)),
                          showmedians=True, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    parts['cmedians'].set_color('#333')
    parts['cmedians'].set_lw(1.5)

    ax.set_xticks(range(len(all_ne)))
    ax.set_xticklabels([str(n) for n in all_ne], rotation=45, ha='right')
    ax.set_xlabel('N_E')
    ax.set_title(label, color='#0b0b0b')

axes[0].set_ylabel('Selectivity index S (0 = uniform, 1 = single class)')
plt.suptitle('Distribution of per-neuron selectivity index by morphology and N_E',
             fontsize=11, color='#0b0b0b', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'sel_sweep_fig3_si_distributions.png', bbox_inches='tight', dpi=200)
plt.show()